In [ ]:
!pip install docling

In [ ]:
import os
import json
from pathlib import Path
from google.colab import files, userdata
from openai import OpenAI
from docling.document_converter import DocumentConverter

# Initialize API client and retrieve credentials
HF_TOKEN = userdata.get('hfprov')
hf_client = OpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=HF_TOKEN
)

# Upload the safety document to the Colab environment
uploaded = files.upload()
filename = list(uploaded.keys())[0]

# Convert the uploaded document into Markdown format
converter = DocumentConverter()
result = converter.convert(filename)
full_markdown_text = result.document.export_to_markdown()

# Define a function to split long text into smaller segments for processing
def chunk_text(text, max_chars=4000):
    chunks = []
    for i in range(0, len(text), max_chars):
        chunks.append(text[i : i + max_chars])
    return chunks

# Prepare text chunks and initialize the central data storage
chunks = chunk_text(full_markdown_text)
all_rules = {
    "PPE_list": set(),
    "Fall_list": {"active": "No", "reason": ""},
    "Heat_list": {"active": "No", "reason": ""}
}

# Iterate through text chunks and extract safety regulations using the AI model
for i, chunk in enumerate(chunks):
    prompt = f"""Analyze the safety document text below and extract specific safety regulations into JSON format.

    TASK 1: PPE Extraction
    - Extract ONLY a list of physical safety equipment (e.g., "Hard Hat", "Safety Boots").
    - Do NOT extract behavioral rules, colors, sizes, or descriptions.
    - Extract only mandatory equipment for employees.

    TASK 2: Fall & Heat Monitoring
    - If the text requires monitoring for falls/heights or fire/smoke/heat, set "active" to "Yes".
    - Provide the direct quote from the PDF as the "reason".

    TEXT SEGMENT:
    {chunk}

    INSTRUCTIONS:
    - STEP A (Normalization): Convert every extracted string to lowercase.
    - STEP B (Singularization): Convert plural items to singular (e.g., "helmets" to "helmet").
    - STEP C (Trimming): Remove any leading or trailing whitespace.
    - STEP D (Deduplication): Remove all duplicate entries.
    - Only return standardized PPE terms. Do not return variations of the same equipment.
    - Return ONLY valid JSON.
    - Schema:
    {{
      "PPE_list": ["item1", "item2"],
      "Fall_list": {{"active": "Yes/No", "reason": "proof text"}},
      "Heat_list": {{"active": "Yes/No", "reason": "proof text"}}
    }}
    """

    try:
        completion = hf_client.chat.completions.create(
            model="openai/gpt-oss-20b:groq",
            messages=[{"role": "user", "content": prompt}],
            response_format={ "type": "json_object" }
        )

        data = json.loads(completion.choices[0].message.content)

        # Merge extracted PPE items and update risk flags if a hazard is detected
        if "PPE_list" in data and isinstance(data["PPE_list"], list):
            all_rules["PPE_list"].update(data["PPE_list"])

        for key in ["Fall_list", "Heat_list"]:
            if key in data and str(data[key].get("active")).lower() == "yes":
                all_rules[key]["active"] = "Yes"
                all_rules[key]["reason"] = data[key].get("reason", "")

    except Exception:
        continue

# Format the final results and save them to a JSON file
final_output = {
    "PPE_list": sorted(list(all_rules["PPE_list"])),
    "Fall_list": all_rules["Fall_list"],
    "Heat_list": all_rules["Heat_list"]
}

json_filename = f"{filename}_safety_config.json"
with open(json_filename, "w") as f:
    json.dump(final_output, f, indent=4)

# Provide the processed file for download
print(f"\nProcessing complete. File ready for download: {json_filename}")
files.download(json_filename)